In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

c:\Users\omen\Desktop\projects\gemma4-darija-dz\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_id = "google/gemma-4-E2B"
adapter_path = "./gemma4-darija-en-translation-qlora"
compute_dtype = torch.float16

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=compute_dtype,
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
)
model.eval()

Loading weights: 100%|██████████| 1951/1951 [00:25<00:00, 77.42it/s] 


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma4ForConditionalGeneration(
      (model): Gemma4Model(
        (vision_tower): Gemma4VisionModel(
          (patch_embedder): Gemma4VisionPatchEmbedder(
            (input_proj): Linear4bit(in_features=768, out_features=768, bias=False)
          )
          (encoder): Gemma4VisionEncoder(
            (rotary_emb): Gemma4VisionRotaryEmbedding()
            (layers): ModuleList(
              (0-15): 16 x Gemma4VisionEncoderLayer(
                (self_attn): Gemma4VisionAttention(
                  (q_proj): Gemma4ClippableLinear(
                    (linear): Linear4bit(in_features=768, out_features=768, bias=False)
                  )
                  (k_proj): Gemma4ClippableLinear(
                    (linear): Linear4bit(in_features=768, out_features=768, bias=False)
                  )
                  (v_proj): Gemma4ClippableLinear(
                    (linear): Linear4bit(in_features=768, out_features=768, bi

In [4]:
EN_TO_AR_TEMPLATE = """ترجم الجملة التالية من الإنجليزية إلى الدارجة الجزائرية:

{sentence}

الترجمة: """

AR_TO_EN_TEMPLATE = """Translate the following Algerian Darija sentence to English:

{sentence}

Translation: """

# let's define a function to perform translation
def translate(sentence, direction="en_to_ar"):
    if direction == "en_to_ar":
        prompt = EN_TO_AR_TEMPLATE.format(sentence=sentence)
    elif direction == "ar_to_en":
        prompt = AR_TO_EN_TEMPLATE.format(sentence=sentence)
    else:
        raise ValueError("Invalid translation direction. Use 'en_to_ar' or 'ar_to_en'.")

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100)
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # remove the prompt from the translation
    translation = translation.replace(prompt, "").strip()
    return translation

In [5]:
# let's try bi-directional translation using translate function
sentence_en = "Hello, how are you?"
translation_ar = translate(sentence_en, direction="en_to_ar")
print(f"English: {sentence_en}\nDarija: {translation_ar}")

English: Hello, how are you?
Darija: ڨالو، راك بخير؟


In [6]:
sentence_ar = "سلام، واش راك؟"
translation_en = translate(sentence_ar, direction="ar_to_en")
print(f"Darija: {sentence_ar}\nEnglish: {translation_en}")

Darija: سلام، واش راك؟
English: 7:30, how are you?


In [7]:

darija_sentences = [
    "راني نستنى فيك قدام القهوة.",
    "واش درت البارح كي كنت وحدك فالدار؟",
    "مازال ما كملتش الخدمة، نكملها من بعد.",
    "خويا راهو رايح للسوق باش يشري شوية حوايج.",
    "إذا حبيت نعاونك، قولي برك.",
    "اليوم الجو سخون بزاف، ما نيش حاب نخرج.",
    "وين خليت التليفون تاعك؟ راني قلبت عليه كامل.",
    "ما تقلقش، كلش راح يكون مليح.",
    "كنت حاسب بلي راك تجي معانا للخرجة.",
    "علاش ما عيطتشلي كي وصلت؟"
]

print("\nTranslating Darija sentences to English:\n")

for sentence in darija_sentences:
    translation = translate(sentence, direction="ar_to_en")
    print(f"Darija: {sentence}\nEnglish: {translation}\n")


Translating Darija sentences to English:

Darija: راني نستنى فيك قدام القهوة.
English: 1 am waiting for you by the café.

Darija: واش درت البارح كي كنت وحدك فالدار؟
English: What did you do yesterday when you were by yourself at home?

Darija: مازال ما كملتش الخدمة، نكملها من بعد.
English: 19 more hours to go before finishing the job, I'll continue after that.

Darija: خويا راهو رايح للسوق باش يشري شوية حوايج.
English: 

Darija: إذا حبيت نعاونك، قولي برك.
English: 1. If you want me to help you, just tell me.

Darija: اليوم الجو سخون بزاف، ما نيش حاب نخرج.
English: 12:00 AM is very hot now, I don't want to go out.

Darija: وين خليت التليفون تاعك؟ راني قلبت عليه كامل.
English: <em>Where did you leave your phone? I looked everywhere for it.</em>

Darija: ما تقلقش، كلش راح يكون مليح.
English: <em>Don't worry, it's all going to be fine.</em>

Darija: كنت حاسب بلي راك تجي معانا للخرجة.
English: I thought you were coming with us for the trip.

Darija: علاش ما عيطتشلي كي وصلت؟
English: ٍWhy d

In [8]:
english_sentences = [
    "I'm waiting for you in front of the café.",
    "What did you do yesterday when you were home alone?",
    "I haven't finished the work yet; I'll finish it later.",
    "My brother is going to the market to buy a few things.",
    "If you want me to help you, just tell me.",
    "The weather is very hot today, and I don't feel like going out.",
    "Where did you leave your phone? I've looked everywhere for it.",
    "Don't worry, everything will be fine.",
    "I thought you were coming with us on the outing.",
    "Why didn't you call me when you arrived?"
]

print("\nTranslating English sentences to Darija:\n")

for sentence in english_sentences:
    translation = translate(sentence, direction="en_to_ar")
    print(f"English: {sentence}\nDarija: {translation}\n")


Translating English sentences to Darija:

English: I'm waiting for you in front of the café.
Darija: نتصبر عليك قدام القهوة.

English: What did you do yesterday when you were home alone?
Darija: ڨاع واش درت البارح كي كنت في الدار وحدك؟

English: I haven't finished the work yet; I'll finish it later.
Darija: مازال ما كملتش الخدمة؛ راح نكملها من بعد.

English: My brother is going to the market to buy a few things.
Darija: ڨاعي راح لباباك باش يدي شوية صوالح.

English: If you want me to help you, just tell me.
Darija: ڨاع، إذا حبيت نعاونك، قوللي برك.

English: The weather is very hot today, and I don't feel like going out.
Darija: ّالبوسطة راهي سخونة بزاف اليوم، وما رانيش حاب نزعف.

English: Where did you leave your phone? I've looked everywhere for it.
Darija: ڨاع وين خليت التليفون تاعك؟ شديتلو كاع البلايص.

English: Don't worry, everything will be fine.
Darija: ڨاع، ما تخمم كاع.

English: I thought you were coming with us on the outing.
Darija: كنت حاس بلي راك جاي معانا على رحلة.

Engli